# Modelado

En esta etapa se construirá el modelo de clasificación de reseñas utilizando técnicas de Procesamiento de Lenguaje Natural (NLP) y Deep Learning. A partir del conjunto de datos previamente preprocesado, las reseñas serán transformadas mediante la representación numérica **TF-IDF**, permitiendo convertir el texto en vectores adecuados para el entrenamiento de un modelo neuronal.

Posteriormente, se implementará un clasificador utilizando **PyTorch**, compuesto por capas completamente conectadas (*fully connected*), Batch Normalization y Dropout para mejorar la estabilidad del entrenamiento y reducir el sobreajuste. El proceso incluirá la división del conjunto de datos, el entrenamiento del modelo, la evaluación de su desempeño y el análisis de las métricas obtenidas.

El objetivo final es desarrollar un modelo capaz de predecir la calificación (*Rating*) de una reseña a partir de su contenido textual, evaluando su capacidad de generalización sobre datos no vistos.

## Importación de librerías y carga del conjunto de datos

Se importan las bibliotecas necesarias para el modelado y se carga el conjunto de datos previamente preprocesado, el cual será utilizado para la representación numérica mediante TF-IDF y el entrenamiento del modelo de clasificación.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
import numpy as np
import pandas as pd

import torch

from sklearn.model_selection import train_test_split

from src.utils.seed import set_seed

RANDOM_STATE = 42

set_seed(RANDOM_STATE)

In [5]:
DATA_PATH = Path("../data/processed/amazon_reviews_processed.csv")

df = pd.read_csv(DATA_PATH)

display(df.head())

,Text,Rating,Age,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,absolutely wonderful - silky sexy comfortable,4,33,1,0,Initmates,Intimate,Intimates
1,love dress ! sooo pretty . happened find store...,5,34,1,4,General,Dresses,Dresses
2,major design flaw high hope dress really wante...,3,60,0,0,General,Dresses,Dresses
3,"favorite buy ! love , love , love jumpsuit . f...",5,50,1,0,General Petite,Bottoms,Pants
4,flattering shirt shirt flattering due adjustab...,5,47,1,6,General,Tops,Blouses


In [6]:
print(f"Observaciones: {len(df):,}")
print(f"Variables: {df.shape[1]}")

Observaciones: 22,641
Variables: 8


## Representación numérica mediante TF-IDF

Los algoritmos de aprendizaje automático no pueden trabajar directamente con texto, por lo que resulta necesario transformar las reseñas en una representación numérica.

En este proyecto se emplea **TF-IDF (Term Frequency - Inverse Document Frequency)**, una técnica que pondera la importancia de cada palabra dentro de un documento considerando también su frecuencia en todo el corpus. De esta forma, los términos muy frecuentes reciben un menor peso, mientras que aquellos más representativos adquieren mayor relevancia.

Esta representación resulta más adecuada que un conteo simple de palabras (*Bag of Words*), ya que permite reducir la influencia de términos poco informativos y mejorar la capacidad discriminativa del modelo de clasificación.

In [7]:
from src.features.tfidf import build_tfidf

MAX_FEATURES = 5000

vectorizer = build_tfidf(MAX_FEATURES)

X = df["Text"]
y = df["Rating"]

X_tfidf = vectorizer.fit_transform(X)

print(f"Número de documentos: {X_tfidf.shape[0]:,}")
print(f"Número de características: {X_tfidf.shape[1]:,}")


Número de documentos: 22,641
Número de características: 5,000


In [10]:
display(
    pd.Series(feature_names)
    .sample(20, random_state=RANDOM_STATE)
    .to_frame(name="Términos del vocabulario")
)

NameError: name 'feature_names' is not defined

### Conclusiones

Las reseñas fueron transformadas exitosamente a una representación numérica mediante TF-IDF, obteniendo una matriz de **22.641 documentos** y un vocabulario de **5.000 características**. Esta representación reduce la influencia de palabras muy frecuentes y conserva los términos más representativos del corpus, proporcionando una entrada adecuada para el entrenamiento del modelo de clasificación.

Se utilizará un vocabulario máximo de 5.000 términos, incluyendo unigramas y bigramas. Además, se eliminarán palabras extremadamente raras o excesivamente frecuentes para mejorar la calidad de la representación numérica.

## División del conjunto de datos

Antes de entrenar el modelo es necesario dividir el conjunto de datos en subconjuntos independientes.

En este proyecto se utilizará una división estratificada para conservar la distribución original de las clases (Rating) en cada partición. De esta manera, tanto el entrenamiento como la validación y la evaluación final se realizan sobre muestras representativas del problema.

In [11]:
from src.models.dataset import split_dataset

(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
) = split_dataset(
    X_tfidf,
    y,
    random_state=RANDOM_STATE,
)

print(f"Entrenamiento: {X_train.shape[0]:,}")
print(f"Validación:    {X_val.shape[0]:,}")
print(f"Prueba:        {X_test.shape[0]:,}")

Entrenamiento: 15,848
Validación:    2,264
Prueba:        4,529


In [12]:
distribution = pd.DataFrame({
    "Original": y.value_counts(normalize=True).sort_index(),
    "Train": y_train.value_counts(normalize=True).sort_index(),
    "Validation": y_val.value_counts(normalize=True).sort_index(),
    "Test": y_test.value_counts(normalize=True).sort_index(),
})

display((distribution * 100).round(2))

,Original,Train,Validation,Test
Rating,,,,
1,3.63,3.63,3.62,3.62
2,6.84,6.84,6.85,6.84
3,12.47,12.47,12.46,12.48
4,21.68,21.67,21.69,21.68
5,55.39,55.39,55.39,55.38


### Conclusiones

El conjunto de datos fue dividido en entrenamiento, validación y prueba utilizando un muestreo estratificado. Esta estrategia conserva la distribución original de las clases en cada subconjunto, permitiendo entrenar el modelo y evaluar su capacidad de generalización sobre datos no utilizados durante el aprendizaje.

## Preparación de los datos para PyTorch

La representación TF-IDF obtenida anteriormente se convierte a tensores de PyTorch para poder ser utilizada durante el entrenamiento de la red neuronal.

Posteriormente se crean los `DataLoader`, que permiten recorrer el conjunto de entrenamiento, validación y prueba en mini-batches, mejorando la eficiencia del entrenamiento y facilitando el uso del descenso por gradiente.

In [24]:
from src.models.dataset import (
    create_tensor_dataset,
    create_dataloader,
)

train_dataset = create_tensor_dataset(X_train, y_train)

val_dataset = create_tensor_dataset(X_val, y_val)

test_dataset = create_tensor_dataset(X_test, y_test)

BATCH_SIZE = 64

train_loader = create_dataloader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = create_dataloader(
    val_dataset,
    batch_size=BATCH_SIZE,
)

test_loader = create_dataloader(
    test_dataset,
    batch_size=BATCH_SIZE,
)

print(f"Batch size: {BATCH_SIZE}")

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Número de muestras: {len(train_dataset)}")

Batch size: 64
Train batches: 248
Validation batches: 36
Test batches: 71
Número de muestras: 15848


In [25]:
features, label = train_dataset[0]

print(features.shape)

print(label)

torch.Size([5000])
tensor(4)


### Conclusiones

Los conjuntos de entrenamiento, validación y prueba fueron convertidos correctamente a tensores de PyTorch y encapsulados en `DataLoader`. Esta estructura permitirá alimentar la red neuronal mediante mini-batches durante el entrenamiento, optimizando el uso de memoria y facilitando la actualización de los parámetros del modelo.

## Implementación de la red neuronal

Se implementa una red neuronal multicapa utilizando PyTorch mediante una clase `TextClassifier`, heredada de `nn.Module`.

La arquitectura recibe como entrada los vectores TF-IDF generados previamente y está compuesta por dos capas densas con Batch Normalization y Dropout, siguiendo las especificaciones de la consigna. Finalmente, la capa de salida produce las probabilidades asociadas a cada una de las cinco clases del problema de clasificación.

In [40]:
from src.models.classifier import TextClassifier

model = TextClassifier()

print(model)

TextClassifier(
  (fc1): Linear(in_features=5000, out_features=256, bias=True)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=256, out_features=5, bias=True)
)


### Batch Normalization

Se incorpora una capa de normalización por lotes (Batch Normalization) inmediatamente después de la primera capa densa. Esta técnica normaliza las activaciones durante el entrenamiento, reduciendo el desplazamiento interno de las distribuciones (internal covariate shift), lo que favorece una convergencia más estable y rápida del modelo. Además, contribuye a mejorar la capacidad de generalización y forma parte de la arquitectura solicitada para este proyecto.

### Función de activación ReLU

Se incorpora la función de activación ReLU (Rectified Linear Unit) después de la normalización por lotes. Esta función introduce no linealidad en la red neuronal, permitiendo que el modelo aprenda relaciones más complejas entre las características TF-IDF y las clases objetivo. Además de su simplicidad computacional, ReLU ayuda a acelerar el entrenamiento y reduce el problema del desvanecimiento del gradiente en comparación con otras funciones de activación.

### Arquitectura del modelo

La arquitectura incorpora una primera capa completamente conectada de 256 neuronas, seguida por Batch Normalization, la función de activación ReLU y una capa Dropout con probabilidad de 0.5 para reducir el sobreajuste. Finalmente, una segunda capa densa proyecta la representación aprendida hacia las cinco clases correspondientes a las calificaciones del conjunto de datos.

In [41]:
from src.models.classifier import TextClassifier

model = TextClassifier()

features, labels = next(iter(train_loader))

print(features.shape)
print(labels.shape)

outputs = model(features)

print(outputs.shape)

print(outputs[0])

torch.Size([64, 5000])
torch.Size([64])
torch.Size([64, 5])
tensor([ 1.0441, -0.3657, -0.2763, -0.6698, -0.7492],
       grad_fn=<SelectBackward0>)


### Propagación hacia adelante (Forward Pass)

Se implementa el método forward(), encargado de definir el flujo de datos dentro de la red neuronal. Cada lote de vectores TF-IDF atraviesa secuencialmente una capa densa, Batch Normalization, la función de activación ReLU, una capa Dropout y, finalmente, una segunda capa densa que produce los logits correspondientes a las cinco clases. No se aplica una función Softmax en la salida, ya que CrossEntropyLoss incorpora esta operación internamente durante el entrenamiento.

In [42]:
import torch

class_counts = y_train.value_counts().sort_index()

print(class_counts)

Rating
1     575
2    1084
3    1976
4    3435
5    8778
Name: count, dtype: int64


In [43]:
class_weights = len(y_train) / (
    len(class_counts) * class_counts
)

class_weights = torch.tensor(
    class_weights.values,
    dtype=torch.float32,
)

print(class_weights)

tensor([5.5123, 2.9240, 1.6040, 0.9227, 0.3611])


### Función de pérdida

Debido al desbalance presente en las clases del conjunto de datos, se utiliza la función de pérdida CrossEntropyLoss con pesos por clase. Estos pesos se calculan de manera inversamente proporcional a la frecuencia de cada categoría, otorgando una mayor penalización a los errores cometidos sobre las clases minoritarias. Esta estrategia busca reducir el sesgo del modelo hacia la clase predominante y mejorar el desempeño global de la clasificación.

In [44]:
from src.models.train import create_optimizer

# -----------------------------
# Pesos de las clases
# -----------------------------

class_counts = y_train.value_counts().sort_index()

class_weights = (
    len(y_train)
    / (len(class_counts) * class_counts)
)

class_weights = torch.tensor(
    class_weights.values,
    dtype=torch.float32,
)

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights,
)

# -----------------------------
# Optimizador
# -----------------------------

LEARNING_RATE = 1e-3

optimizer = create_optimizer(
    model,
    learning_rate=LEARNING_RATE,
)

print(criterion)
print(optimizer)

CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


### Configuración del optimizador

Para el entrenamiento de la red neuronal se emplea el optimizador **Adam (Adaptive Moment Estimation)**, uno de los algoritmos más utilizados en tareas de aprendizaje profundo debido a su capacidad para adaptar automáticamente la tasa de aprendizaje de cada parámetro durante el entrenamiento.

Se utiliza una tasa de aprendizaje inicial de **0.001**, valor que representa un equilibrio adecuado entre velocidad de convergencia y estabilidad. Adam combina las ventajas de los algoritmos Momentum y RMSProp, permitiendo un aprendizaje eficiente incluso en espacios de alta dimensionalidad como los generados por la representación TF-IDF.

Junto con la función de pérdida **CrossEntropyLoss**, Adam será el encargado de actualizar iterativamente los pesos de la red neuronal mediante retropropagación del error, buscando minimizar la pérdida sobre el conjunto de entrenamiento.

In [45]:
import importlib
import src.models.train as trainer

importlib.reload(trainer)

<module 'src.models.train' from 'c:\\Users\\lauta\\Desktop\\DataScience\\amazon_reviews_nlp\\src\\models\\train.py'>

In [46]:
from src.models.train import (
    create_optimizer,
    train_model,
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(DEVICE)

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights.to(DEVICE)
)

optimizer = create_optimizer(
    model,
    learning_rate=1e-3,
)

In [47]:
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
    epochs=30,
    patience=5,
)

Epoch 01/30 | Train Loss: 1.2616 | Val Loss: 1.1144 | Train Acc: 0.5358 | Val Acc: 0.5720
Epoch 02/30 | Train Loss: 0.8077 | Val Loss: 1.2093 | Train Acc: 0.7215 | Val Acc: 0.5799
Epoch 03/30 | Train Loss: 0.5273 | Val Loss: 1.4053 | Train Acc: 0.8028 | Val Acc: 0.5813
Epoch 04/30 | Train Loss: 0.3349 | Val Loss: 1.5600 | Train Acc: 0.8681 | Val Acc: 0.5861
Epoch 05/30 | Train Loss: 0.2180 | Val Loss: 1.8125 | Train Acc: 0.9068 | Val Acc: 0.5981
Epoch 06/30 | Train Loss: 0.1595 | Val Loss: 1.9541 | Train Acc: 0.9293 | Val Acc: 0.5928

Early Stopping activado.


### Entrenamiento del modelo

Se implementó un ciclo completo de entrenamiento utilizando PyTorch. Durante cada época, el modelo procesa los lotes del conjunto de entrenamiento, calcula la pérdida mediante **CrossEntropyLoss**, actualiza sus parámetros utilizando el optimizador **Adam** y registra tanto la pérdida como la exactitud obtenida.

Al finalizar cada época se evalúa el desempeño sobre el conjunto de validación, permitiendo monitorear la capacidad de generalización del modelo. Para evitar el sobreajuste se implementó la técnica de **Early Stopping**, que detiene automáticamente el entrenamiento cuando la pérdida de validación no mejora durante cinco épocas consecutivas. Además, se conservan los pesos correspondientes al mejor desempeño observado durante el entrenamiento, asegurando que el modelo final corresponda a la mejor versión obtenida.

In [52]:
import joblib
import torch
from pathlib import Path

# ---------------------------------
# Crear directorios si no existen
# ---------------------------------

MODEL_DIR = Path("../models")

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ---------------------------------
# Guardar pesos del modelo
# ---------------------------------

torch.save(
    model.state_dict(),
    MODEL_DIR / "text_classifier.pth",
)

In [53]:
# ---------------------------------
# Guardar vectorizador TF-IDF
# ---------------------------------

joblib.dump(
    vectorizer,
    MODEL_DIR / "tfidf_vectorizer.pkl",
)

print("Modelo guardado correctamente.")
print("Vectorizador guardado correctamente.")

Modelo guardado correctamente.
Vectorizador guardado correctamente.


In [54]:
# ---------------------------------
# Guardar historial de entrenamiento
# ---------------------------------

joblib.dump(
    history,
    MODEL_DIR / "training_history.pkl",
)

print("Historial guardado correctamente.")

Historial guardado correctamente.


In [55]:
print(f"Mejor pérdida de validación: {min(history['val_loss']):.4f}")
print(f"Mejor accuracy de validación: {max(history['val_acc']):.4f}")

Mejor pérdida de validación: 1.1144
Mejor accuracy de validación: 0.5981


## Conclusión

En este notebook se desarrolló la etapa completa de modelado para la tarea de clasificación automática de reseñas de productos. A partir del conjunto de datos previamente preprocesado y representado mediante TF-IDF, se construyó una red neuronal multicapa utilizando PyTorch.

La arquitectura implementada incorpora dos capas densas, Batch Normalization, función de activación ReLU y una capa Dropout con probabilidad de 0.5 para reducir el riesgo de sobreajuste. El entrenamiento se realizó utilizando el optimizador Adam y la función de pérdida CrossEntropyLoss ponderada según la distribución de clases, con el objetivo de mitigar el desbalance presente en el dataset.

Asimismo, se implementó un mecanismo de Early Stopping basado en la pérdida de validación, permitiendo detener automáticamente el entrenamiento cuando el modelo deja de mejorar y conservando los pesos correspondientes al mejor desempeño obtenido.

Finalmente, se almacenaron tanto los pesos del modelo entrenado como el vectorizador TF-IDF y el historial del entrenamiento, permitiendo desacoplar el proceso de entrenamiento de la etapa de evaluación. En el siguiente notebook se analizarán las métricas de desempeño obtenidas sobre el conjunto de prueba, junto con una interpretación de los resultados y posibles líneas de mejora para el modelo desarrollado.